# FinChart-R2 — Phase 2B Pilot SFT

Train `unsloth/Qwen3-VL-4B-Instruct-unsloth-bnb-4bit` with QLoRA using:

```text
phase2a_pilot_500_v3_train_clean.jsonl
```

Expected clean pilot set: **408 samples** from the 500-sample Phase 2A run.

The frozen Phase 1 evaluation set (`ChartQA val[0:500]`) is never used for training.

## 1. Install dependencies

In [ ]:
!pip install -q -U unsloth unsloth_zoo datasets trl transformers accelerate bitsandbytes

## 2. Imports and GPU check

In [ ]:
import os
import json
import random
from pathlib import Path

import torch
import pandas as pd
from datasets import load_dataset, Dataset

SEED = 3407
random.seed(SEED)
torch.manual_seed(SEED)

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("GPU runtime required.")

print("GPU:", torch.cuda.get_device_name(0))
print(
    "VRAM:",
    round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3,
        2,
    ),
    "GB",
)

## 3. Mount Drive and define paths

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive")
PROJECT_DIR = ROOT / "FinChart-R2"
DATA_DIR = PROJECT_DIR / "data"

PHASE2B_DIR = PROJECT_DIR / "phase2b"
RUN_DIR = PHASE2B_DIR / "pilot_sft_408"
ADAPTER_DIR = RUN_DIR / "adapter"
CHECKPOINT_DIR = RUN_DIR / "checkpoints"

for p in [PHASE2B_DIR, RUN_DIR, ADAPTER_DIR, CHECKPOINT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

TRAIN_JSONL = (
    DATA_DIR
    / "phase2a_pilot_500_v3_train_clean.jsonl"
)

print("Train JSONL:", TRAIN_JSONL)
print("Run dir:", RUN_DIR)

## 4. Load strict-clean Phase 2A data

In [ ]:
if not TRAIN_JSONL.exists():
    raise FileNotFoundError(TRAIN_JSONL)

records = []

with open(TRAIN_JSONL, "r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue

        try:
            records.append(json.loads(line))
        except json.JSONDecodeError as exc:
            raise ValueError(
                f"Invalid JSON at line {line_no}: {exc}"
            )

print("Loaded:", len(records))

if records:
    print("Columns:")
    print(sorted(records[0].keys()))

## 5. Validate training artifact before touching the model

In [ ]:
EXPECTED_CLEAN = 408

ALLOWED_TASKS = {
    "numerical_reasoning",
    "visual_grounding",
    "counting",
    "logical_reasoning",
}

assert records, "Empty training file."

if len(records) != EXPECTED_CLEAN:
    print(
        f"WARNING: expected {EXPECTED_CLEAN} rows, "
        f"found {len(records)}."
    )

indices = []

for i, row in enumerate(records):
    assert "dataset_index" in row, f"Row {i}: missing dataset_index"
    assert str(row.get("question", "")).strip(), f"Row {i}: missing question"

    has_target = bool(str(row.get("sft_target", "")).strip())
    has_answer = bool(
        str(
            row.get(
                "dataset_answer",
                row.get("canonical_answer", ""),
            )
        ).strip()
    )

    assert has_target or has_answer, (
        f"Row {i}: missing both sft_target and answer"
    )

    task = row.get(
        "teacher_task_type",
        row.get("task_type"),
    )

    if task is not None:
        assert task in ALLOWED_TASKS, (
            f"Row {i}: invalid task={task}"
        )

    indices.append(int(row["dataset_index"]))

assert len(indices) == len(set(indices)), (
    "Duplicate dataset_index values detected."
)

print("Artifact validation: PASS")

## 6. Inspect task distribution

In [ ]:
df = pd.DataFrame(records)

task_col = (
    "teacher_task_type"
    if "teacher_task_type" in df.columns
    else "task_type"
)

if task_col in df.columns:
    stats = pd.DataFrame({
        "count": df[task_col].value_counts(),
        "share": df[task_col].value_counts(normalize=True),
    })
    display(stats)

display(df.head(10))

## 7. Load ChartQA train images

The clean JSONL stores `dataset_index`. Images are reconstructed from the original `ChartQA train` split.

In [ ]:
DATASET_NAME = "HuggingFaceM4/ChartQA"

chartqa_train = load_dataset(
    DATASET_NAME,
    split="train",
)

print("ChartQA train:", len(chartqa_train))

assert min(indices) >= 0
assert max(indices) < len(chartqa_train)

print("All image indices valid.")

## 8. Build structured SFT targets

In [ ]:
def clean_text(x):
    return "" if x is None else str(x).strip()


def build_target(row):
    existing = clean_text(row.get("sft_target"))

    if existing:
        return existing

    lines = []

    series = clean_text(row.get("target_series"))
    category = clean_text(row.get("target_category"))

    if series:
        lines.append(f"Target series: {series}")

    if category:
        lines.append(f"Target category: {category}")

    values = row.get("relevant_values")

    if isinstance(values, list) and values:
        lines.append(
            "Relevant values: "
            + ", ".join(map(str, values))
        )

    operation = clean_text(row.get("operation"))

    if operation and operation != "none":
        lines.append(f"Operation: {operation}")

    calculation = clean_text(row.get("calculation"))

    if calculation:
        lines.append(f"Calculation: {calculation}")

    answer = clean_text(
        row.get(
            "dataset_answer",
            row.get("canonical_answer"),
        )
    )

    if not answer:
        raise ValueError(
            f"No final answer for dataset_index={row['dataset_index']}"
        )

    lines.append(f"Answer: {answer}")

    return "\n".join(lines)


for row in records:
    row["final_sft_target"] = build_target(row)

print(records[0]["final_sft_target"])

## 9. Convert to multimodal conversations

In [ ]:
def to_conversation(row):
    idx = int(row["dataset_index"])

    return {
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "image": chartqa_train[idx]["image"],
                    },
                    {
                        "type": "text",
                        "text": clean_text(row["question"]),
                    },
                ],
            },
            {
                "role": "assistant",
                "content": [
                    {
                        "type": "text",
                        "text": row["final_sft_target"],
                    }
                ],
            },
        ]
    }


conversation_records = [
    to_conversation(row)
    for row in records
]

train_dataset = Dataset.from_list(
    conversation_records
)

print(train_dataset)

## 10. Visual sanity check

In [ ]:
import matplotlib.pyplot as plt

for row in records[:4]:
    idx = int(row["dataset_index"])

    plt.figure(figsize=(8, 5))
    plt.imshow(chartqa_train[idx]["image"])
    plt.axis("off")
    plt.title(row["question"], fontsize=10)
    plt.show()

    print("TARGET")
    print(row["final_sft_target"])
    print("=" * 90)

## 11. Load Qwen3-VL-4B in 4-bit

In [ ]:
from unsloth import FastVisionModel

BASE_MODEL = (
    "unsloth/"
    "Qwen3-VL-4B-Instruct-unsloth-bnb-4bit"
)

MAX_SEQ_LENGTH = 2048

model, processor = FastVisionModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

print("Loaded:", BASE_MODEL)

## 12. Attach QLoRA adapters

In [ ]:
model = FastVisionModel.get_peft_model(
    model,

    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,

    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",

    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)

model.print_trainable_parameters()

## 13. Training configuration

Pilot settings:

```text
records                 408
batch size              1
gradient accumulation   8
effective batch         8
epochs                  2
learning rate           1e-4
LoRA r / alpha          16 / 16
```

This is deliberately conservative because the pilot dataset is small.

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

FastVisionModel.for_training(model)

training_args = SFTConfig(
    output_dir=str(CHECKPOINT_DIR),

    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,

    num_train_epochs=2,
    learning_rate=1e-4,

    warmup_ratio=0.05,
    weight_decay=0.01,
    lr_scheduler_type="cosine",

    optim="adamw_8bit",

    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),

    logging_steps=1,
    save_strategy="epoch",

    seed=SEED,
    report_to="none",

    remove_unused_columns=False,
    dataset_text_field="",
    dataset_kwargs={
        "skip_prepare_dataset": True,
    },
    max_length=MAX_SEQ_LENGTH,
)

trainer = SFTTrainer(
    model=model,
    processing_class=processor,
    data_collator=UnslothVisionDataCollator(
        model,
        processor,
    ),
    train_dataset=train_dataset,
    args=training_args,
)

print("Trainer ready.")

# 14. Train

In [ ]:
torch.cuda.empty_cache()

trainer_stats = trainer.train()

print("Training complete.")
print(trainer_stats)

## 15. Save adapter + processor

In [ ]:
model.save_pretrained(
    str(ADAPTER_DIR)
)

processor.save_pretrained(
    str(ADAPTER_DIR)
)

print("Saved:", ADAPTER_DIR)

## 16. Save training metrics

In [ ]:
metrics = dict(trainer_stats.metrics)

metrics.update({
    "base_model": BASE_MODEL,
    "training_records": len(records),
    "phase2a_source": TRAIN_JSONL.name,
    "epochs": 2,
    "learning_rate": 1e-4,
    "lora_r": 16,
    "lora_alpha": 16,
    "phase1_validation_used_for_training": False,
})

METRICS_JSON = (
    RUN_DIR
    / "pilot_sft_training_metrics.json"
)

METRICS_JSON.write_text(
    json.dumps(
        metrics,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

print(json.dumps(metrics, indent=2, default=str))
print("Saved:", METRICS_JSON)

## 17. Quick post-training sanity inference

This is not the frozen Phase 1 evaluation. It only checks whether the trained adapter produces reasonable structured output.

In [ ]:
FastVisionModel.for_inference(model)

def infer_one(row, max_new_tokens=256):
    idx = int(row["dataset_index"])
    image = chartqa_train[idx]["image"]

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image,
                },
                {
                    "type": "text",
                    "text": row["question"],
                },
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = processor(
        text=text,
        images=image,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
        )

    new_tokens = outputs[
        :,
        inputs["input_ids"].shape[1]:
    ]

    return processor.batch_decode(
        new_tokens,
        skip_special_tokens=True,
    )[0]


for row in records[:3]:
    print("QUESTION")
    print(row["question"])
    print()

    print("TARGET")
    print(row["final_sft_target"])
    print()

    print("MODEL")
    print(infer_one(row))
    print("=" * 100)

In [ ]:
# ============================================================
# EXTRA GENERALIZATION TEST — 500 UNSEEN TRAIN SAMPLES
# ============================================================

import random
import re
import math

EXTRA_TEST_N = 500
EXTRA_TEST_SEED = 2026

# ------------------------------------------------------------
# IMPORTANT:
# Phase 2A pilot source was built from ChartQA train[0:500].
# Exclude the ENTIRE original 500-source pool,
# not only the 408 samples that survived cleaning.
# ------------------------------------------------------------

OLD_PHASE2A_SOURCE_INDICES = set(range(500))

# Safety: also exclude every index actually used for SFT.
OLD_SFT_INDICES = {
    int(row["dataset_index"])
    for row in records
}

EXCLUDED_INDICES = (
    OLD_PHASE2A_SOURCE_INDICES
    | OLD_SFT_INDICES
)

candidate_indices = [
    i
    for i in range(len(chartqa_train))
    if i not in EXCLUDED_INDICES
]

rng = random.Random(EXTRA_TEST_SEED)

extra_test_indices = rng.sample(
    candidate_indices,
    k=min(EXTRA_TEST_N, len(candidate_indices)),
)

assert not (
    set(extra_test_indices)
    & EXCLUDED_INDICES
)

print("Old Phase 2A source:", len(OLD_PHASE2A_SOURCE_INDICES))
print("Actually used for SFT:", len(OLD_SFT_INDICES))
print("New test samples:", len(extra_test_indices))
print(
    "Overlap:",
    len(
        set(extra_test_indices)
        & EXCLUDED_INDICES
    ),
)

print(
    "Index range:",
    min(extra_test_indices),
    "→",
    max(extra_test_indices),
)


# ------------------------------------------------------------
# Helper for ChartQA fields
# ------------------------------------------------------------

def unwrap_chartqa_answer(value):
    if isinstance(value, (list, tuple)):
        if len(value) == 1:
            return value[0]

        return value

    return value


def get_chartqa_question(example):
    return str(
        example.get(
            "query",
            example.get("question", ""),
        )
    ).strip()


def get_chartqa_answer(example):
    value = example.get(
        "label",
        example.get("answer", ""),
    )

    return str(
        unwrap_chartqa_answer(value)
    ).strip()


# Inspect a few
for idx in extra_test_indices[:5]:

    ex = chartqa_train[idx]

    print()
    print("INDEX:", idx)
    print("Q:", get_chartqa_question(ex))
    print("GT:", get_chartqa_answer(ex))

In [ ]:
# ============================================================
# RUN EXTRA 500-SAMPLE GENERALIZATION TEST
# ============================================================

from tqdm.auto import tqdm
import pandas as pd
import torch
import re


# ------------------------------------------------------------
# Extract final answer from structured model output
# ------------------------------------------------------------

def extract_final_answer(text):
    if text is None:
        return ""

    text = str(text).strip()

    # Prefer explicit:
    # Answer: ...
    matches = re.findall(
        r"(?:^|\\n)Answer\\s*:\\s*(.+)",
        text,
        flags=re.IGNORECASE,
    )

    if matches:
        return matches[-1].strip()

    # fallback = final non-empty line
    lines = [
        x.strip()
        for x in text.splitlines()
        if x.strip()
    ]

    return (
        lines[-1]
        if lines
        else text
    )


def normalize_eval_answer(value):
    text = str(value).strip().lower()

    text = text.replace(",", "")

    text = re.sub(
        r"\\s+",
        " ",
        text,
    )

    return text.strip(
        " .,:;\\n\\t"
    )


def parse_eval_number(value):
    text = normalize_eval_answer(value)

    is_percent = text.endswith("%")

    if is_percent:
        text = text[:-1].strip()

    text = (
        text
        .replace("$", "")
        .replace("€", "")
        .replace("£", "")
    )

    try:
        return float(text), is_percent
    except Exception:
        return None


def answers_match(gt, pred, tolerance=1e-5):

    gt_n = normalize_eval_answer(gt)
    pred_n = normalize_eval_answer(pred)

    # direct normalized string
    if gt_n == pred_n:
        return True

    g = parse_eval_number(gt)
    p = parse_eval_number(pred)

    if g is None or p is None:
        return False

    gv, gpct = g
    pv, ppct = p

    # direct numeric equality
    if math.isclose(
        gv,
        pv,
        rel_tol=tolerance,
        abs_tol=tolerance,
    ):
        return True

    # 0.77 ↔ 77%
    if gpct != ppct:

        if gpct:
            return math.isclose(
                gv / 100,
                pv,
                rel_tol=tolerance,
                abs_tol=tolerance,
            )

        return math.isclose(
            gv,
            pv / 100,
            rel_tol=tolerance,
            abs_tol=tolerance,
        )

    return False


# ------------------------------------------------------------
# Inference function
# ------------------------------------------------------------

FastVisionModel.for_inference(model)

def infer_chartqa_index(
    idx,
    max_new_tokens=192,
):
    example = chartqa_train[idx]

    image = example["image"]
    question = get_chartqa_question(
        example
    )

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image,
                },
                {
                    "type": "text",
                    "text": question,
                },
            ],
        }
    ]

    prompt = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = processor(
        text=prompt,
        images=image,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():

        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
        )

    generated_ids = output_ids[
        :,
        inputs["input_ids"].shape[1]:
    ]

    full_output = (
        processor.batch_decode(
            generated_ids,
            skip_special_tokens=True,
        )[0]
    )

    return full_output


# ------------------------------------------------------------
# Run
# ------------------------------------------------------------

extra_results = []

for idx in tqdm(
    extra_test_indices,
    desc="Extra unseen test",
):

    example = chartqa_train[idx]

    question = get_chartqa_question(
        example
    )

    gt = get_chartqa_answer(
        example
    )

    try:

        model_output = (
            infer_chartqa_index(idx)
        )

        predicted_answer = (
            extract_final_answer(
                model_output
            )
        )

        correct = answers_match(
            gt,
            predicted_answer,
        )

        error = None

    except Exception as exc:

        model_output = ""
        predicted_answer = ""
        correct = False
        error = (
            f"{type(exc).__name__}: "
            f"{exc}"
        )

    extra_results.append({
        "dataset_index":
            idx,

        "question":
            question,

        "gt_answer":
            gt,

        "predicted_answer":
            predicted_answer,

        "correct":
            correct,

        "model_output":
            model_output,

        "error":
            error,
    })


extra_results_df = pd.DataFrame(
    extra_results
)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

valid_df = extra_results_df[
    extra_results_df["error"].isna()
]

accuracy = (
    valid_df["correct"].mean()
    if len(valid_df)
    else 0
)

print()
print("=" * 70)
print("EXTRA GENERALIZATION TEST")
print("=" * 70)

print(
    "Samples:",
    len(extra_results_df),
)

print(
    "Successful inference:",
    len(valid_df),
)

print(
    "Correct:",
    int(
        valid_df["correct"].sum()
    ),
)

print(
    f"Preliminary accuracy: "
    f"{accuracy:.2%}"
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

EXTRA_TEST_CSV = (
    RUN_DIR
    / "pilot_sft_extra_500_unseen_train.csv"
)

extra_results_df.to_csv(
    EXTRA_TEST_CSV,
    index=False,
)

print()
print(
    "Saved:",
    EXTRA_TEST_CSV,
)


# ------------------------------------------------------------
# Inspect
# ------------------------------------------------------------

display(
    extra_results_df[
        [
            "dataset_index",
            "question",
            "gt_answer",
            "predicted_answer",
            "correct",
        ]
    ].head(30)
)

In [ ]:
import pandas as pd
import re
import math
from pathlib import Path

CSV_PATH = Path(
    "/content/drive/MyDrive/FinChart-R2/"
    "phase2b/pilot_sft_408/"
    "pilot_sft_extra_500_unseen_train.csv"
)

df = pd.read_csv(CSV_PATH)

print("Loaded:", len(df))
print("Columns:", df.columns.tolist())


# ============================================================
# FIX ANSWER EXTRACTION
# ============================================================

def extract_final_answer(text):
    if pd.isna(text):
        return ""

    text = str(text).strip()

    # Find the final "Answer: ..."
    matches = re.findall(
        r"^\s*Answer\s*:\s*(.+?)\s*$",
        text,
        flags=re.IGNORECASE | re.MULTILINE,
    )

    if matches:
        return matches[-1].strip()

    # Fallback
    match = re.search(
        r"Answer\s*:\s*(.+)",
        text,
        flags=re.IGNORECASE,
    )

    if match:
        return match.group(1).strip()

    return text.strip()


def normalize_answer(value):
    if pd.isna(value):
        return ""

    text = str(value).strip().lower()

    # Extra safety
    text = re.sub(
        r"^\s*answer\s*:\s*",
        "",
        text,
        flags=re.IGNORECASE,
    )

    text = text.replace(",", "")
    text = re.sub(r"\s+", " ", text)

    return text.strip(" .,:;\n\t")


def parse_number(value):
    text = normalize_answer(value)

    is_percent = text.endswith("%")

    if is_percent:
        text = text[:-1].strip()

    text = (
        text
        .replace("$", "")
        .replace("€", "")
        .replace("£", "")
    )

    try:
        return float(text), is_percent
    except ValueError:
        return None


def answers_match(gt, pred, tolerance=1e-5):

    gt_norm = normalize_answer(gt)
    pred_norm = normalize_answer(pred)

    # Text exact match
    if gt_norm == pred_norm:
        return True

    gt_num = parse_number(gt)
    pred_num = parse_number(pred)

    if gt_num is None or pred_num is None:
        return False

    gv, g_percent = gt_num
    pv, p_percent = pred_num

    # Numeric equality
    if math.isclose(
        gv,
        pv,
        rel_tol=tolerance,
        abs_tol=tolerance,
    ):
        return True

    # Handle 0.77 <-> 77%
    if g_percent != p_percent:

        if g_percent:
            return math.isclose(
                gv / 100.0,
                pv,
                rel_tol=tolerance,
                abs_tol=tolerance,
            )

        return math.isclose(
            gv,
            pv / 100.0,
            rel_tol=tolerance,
            abs_tol=tolerance,
        )

    return False


# ============================================================
# RE-EVALUATE EXISTING OUTPUT
# ============================================================

# Prefer model_output because it contains the complete generation.
# If unavailable, fall back to predicted_answer.
source_col = (
    "model_output"
    if "model_output" in df.columns
    else "predicted_answer"
)

df["predicted_answer_fixed"] = (
    df[source_col]
    .apply(extract_final_answer)
)

df["correct_fixed"] = df.apply(
    lambda row: answers_match(
        row["gt_answer"],
        row["predicted_answer_fixed"],
    ),
    axis=1,
)


# ============================================================
# RESULTS
# ============================================================

correct = int(df["correct_fixed"].sum())
total = len(df)

accuracy = (
    correct / total
    if total
    else 0
)

print()
print("=" * 70)
print("EXTRA GENERALIZATION TEST — RE-EVALUATED")
print("=" * 70)

print("Samples:", total)
print("Correct:", correct)
print("Incorrect:", total - correct)
print(f"Accuracy: {accuracy:.2%}")

print()
print("First 30 results:")

display(
    df[
        [
            "dataset_index",
            "question",
            "gt_answer",
            "predicted_answer_fixed",
            "correct_fixed",
        ]
    ].head(30)
)

In [ ]:
# ============================================================
# PROTOCOL-IDENTICAL FROZEN PHASE 1 EVALUATION ? ChartQA val[0:500]
# ============================================================
# Run this cell after training to produce the final comparable Phase 2B score.
# It intentionally reuses Phase 1's prompt, 64-token generation budget, and matcher.

from datasets import load_dataset
from unsloth import FastVisionModel
from peft import PeftModel
from tqdm.auto import tqdm
import pandas as pd
import torch
import re
import string

FROZEN_EVAL_N = 500
FROZEN_MAX_NEW_TOKENS = 64
EVAL_ADAPTER = ADAPTER_DIR  # Local adapter produced by this notebook.
EVAL_OUTPUT_DIR = PROJECT_DIR / "results" / "vali"
EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = EVAL_OUTPUT_DIR / "sft_val_500.csv"

if not EVAL_ADAPTER.exists():
    raise FileNotFoundError(
        f"Missing local adapter: {EVAL_ADAPTER}. Run the training and save cells first."
    )

# Reload to make the evaluated adapter explicit and reproducible.
del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model, processor = FastVisionModel.from_pretrained(
    BASE_MODEL,
    load_in_4bit=True,
    max_seq_length=2048,
)
model = PeftModel.from_pretrained(model, str(EVAL_ADAPTER))
FastVisionModel.for_inference(model)

chartqa_val = load_dataset("HuggingFaceM4/ChartQA", split="val")
eval_ds = chartqa_val.select(range(FROZEN_EVAL_N))


def unwrap_answer(value):
    if isinstance(value, (list, tuple)) and len(value) == 1:
        return value[0]
    return value


def normalize_phase1_answer(text):
    text = "" if text is None else str(text)
    text = text.lower().strip()
    text = re.sub(r"^\s*final\s+answer\s*:\s*", "", text)
    text = re.sub(r"^\s*answer\s*:\s*", "", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s*/\s*", "/", text)
    return text.strip(string.whitespace + ".,;:!?")


def try_parse_phase1_number(text):
    try:
        return float(
            normalize_phase1_answer(text)
            .replace(",", "")
            .replace("%", "")
            .strip()
        )
    except (ValueError, TypeError):
        return None


def deterministic_phase1_match(prediction, ground_truth, tolerance=1e-6):
    pred = normalize_phase1_answer(prediction)
    gt = normalize_phase1_answer(ground_truth)
    if pred == gt:
        return True
    pred_num = try_parse_phase1_number(pred)
    gt_num = try_parse_phase1_number(gt)
    return (
        pred_num is not None
        and gt_num is not None
        and abs(pred_num - gt_num) <= tolerance
    )


def build_phase1_messages(image, question):
    return [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {
                "type": "text",
                "text": (
                    "Look carefully at the chart and answer the question.\n\n"
                    f"Question: {question}\n\n"
                    "Return only the final answer."
                ),
            },
        ],
    }]


results = []
for index, example in enumerate(tqdm(eval_ds, desc="Frozen Phase 1-compatible eval")):
    question = str(example.get("query", example.get("question", ""))).strip()
    ground_truth = str(
        unwrap_answer(example.get("label", example.get("answer", "")))
    ).strip()
    messages = build_phase1_messages(example["image"], question)
    prompt = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = processor(
        text=prompt,
        images=example["image"],
        return_tensors="pt",
    ).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=FROZEN_MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
        )
    prediction = processor.batch_decode(
        output_ids[:, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )[0].strip()
    results.append({
        "dataset_index": index,
        "question": question,
        "ground_truth": ground_truth,
        "prediction": prediction,
        "deterministic_correct": deterministic_phase1_match(prediction, ground_truth),
    })

eval_df = pd.DataFrame(results)
eval_df.to_csv(CSV_PATH, index=False)
print(f"Frozen Phase 1-compatible accuracy: {eval_df['deterministic_correct'].mean():.2%}")
print(f"Saved: {CSV_PATH}")


In [ ]:
# The preceding cell saves the canonical raw prediction file locally.
# Keep it out of Git; summarize approved metrics in ../../reports/ instead.
assert CSV_PATH.exists(), f"Missing evaluation file: {CSV_PATH}"
print(f"Canonical evaluation file: {CSV_PATH}")


## 18. Pilot completion gate

In [ ]:
adapter_files = list(ADAPTER_DIR.glob("*"))

PILOT_SFT_COMPLETE = (
    len(records) > 0
    and len(adapter_files) > 0
    and METRICS_JSON.exists()
)

print(
    json.dumps(
        {
            "training_records": len(records),
            "adapter_saved": len(adapter_files) > 0,
            "metrics_saved": METRICS_JSON.exists(),
            "phase1_validation_used_for_training": False,
            "pilot_sft_complete": bool(PILOT_SFT_COMPLETE),
        },
        indent=2,
    )
)

# 19. Next step

Load the saved adapter and run the **unchanged Phase 1 evaluator** on:

```text
ChartQA val[0:500]
```

Compare Base vs Pilot SFT on:

- deterministic accuracy
- resolved accuracy
- resolved coverage
- numerical reasoning errors
- visual extraction errors
- counting errors
- logical reasoning errors

Only this frozen evaluation determines whether the Phase 2 SFT hypothesis worked.